In [1]:
import argparse
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import zipfile

In [2]:
class args:
    dataset_dir = "/kaggle/input/datasets/almiraraisa/aptos-2019-224pixels"
    output_dir  = "/kaggle/working/aptos-2019-augmented"
    aug_copies  = 5
    seed        = 35
    image_size  = 224

In [3]:
TRAIN_SPATIAL = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=15),
    transforms.ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2)),
])

In [4]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def load_image(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def save_image(img: Image.Image, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path, format="PNG")


def resolve_fname(name: str) -> str:
    """Ensure the filename ends with .png."""
    return name if name.endswith(".png") else name + ".png"

In [5]:
def process_val_or_test(df: pd.DataFrame, src_img_dir: Path,
                        dst_img_dir: Path, split_name: str) -> pd.DataFrame:
    rows = []
    img_col = "id_code" if "id_code" in df.columns else df.columns[0]
 
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {split_name}"):
        fname = resolve_fname(str(row[img_col]))
        src   = src_img_dir / fname
        dst   = dst_img_dir / fname
 
        if not dst.exists():
            shutil.copy2(src, dst)
 
        rows.append({img_col: Path(fname).stem, "diagnosis": int(row["diagnosis"])})
 
    return pd.DataFrame(rows)

In [6]:
# each train image create an augmented versions. original is kept so model can see clean images
def process_train(df: pd.DataFrame, src_img_dir: Path,
                  dst_img_dir: Path, aug_copies: int, seed: int) -> pd.DataFrame:
    img_col = "id_code" if "id_code" in df.columns else df.columns[0]
    rows = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting train"):
        fname     = resolve_fname(str(row[img_col]))
        stem      = Path(fname).stem
        diagnosis = int(row["diagnosis"])
        src       = src_img_dir / fname
        img       = load_image(src)

        # ── Copy 0: original, unaugmented ──
        dst = dst_img_dir / f"{stem}_orig.png"
        save_image(img, dst)
        rows.append({img_col: f"{stem}_orig", "diagnosis": diagnosis})

        # ── Copies 1 … aug_copies: augmented ──
        for copy_idx in range(1, aug_copies + 1):
            # Seed is a function of (global seed, row index, copy index) so
            # re-running the script always produces the same output files.
            deterministic_seed = seed ^ (idx * 1000 + copy_idx)
            random.seed(deterministic_seed)
            np.random.seed(deterministic_seed)
            # torchvision transforms use Python's random + numpy internally
            aug_img = TRAIN_SPATIAL(img)

            dst = dst_img_dir / f"{stem}_aug{copy_idx:02d}.png"
            save_image(aug_img, dst)
            rows.append({img_col: f"{stem}_aug{copy_idx:02d}", "diagnosis": diagnosis})

    return pd.DataFrame(rows)

In [7]:
def main():
    set_seed(args.seed)

    dataset_dir = Path(args.dataset_dir)
    output_dir  = Path(args.output_dir)
    src_img_dir = dataset_dir / "images"

    dst_img_dir = output_dir / "images"
    dst_img_dir.mkdir(parents=True, exist_ok=True)

    csv_names = {
        "train": "train_split.csv",
        "val":   "val_split.csv",
        "test":  "test_split.csv",
    }


    train_df = pd.read_csv(dataset_dir / csv_names["train"])
    val_df   = pd.read_csv(dataset_dir / csv_names["val"])
    test_df  = pd.read_csv(dataset_dir / csv_names["test"])

    print(f"\nDataset root : {dataset_dir}")
    print(f"Output root  : {output_dir}")
    print(f"Train rows   : {len(train_df)}  →  {len(train_df) * (args.aug_copies + 1)} after augmentation")
    print(f"Val rows     : {len(val_df)}")
    print(f"Test rows    : {len(test_df)}\n")

    aug_train_df = process_train(train_df, src_img_dir, dst_img_dir,
                                 args.aug_copies, args.seed)
    aug_val_df   = process_val_or_test(val_df,  src_img_dir, dst_img_dir, "val")
    aug_test_df  = process_val_or_test(test_df, src_img_dir, dst_img_dir, "test")

    aug_train_df.to_csv(output_dir / "train_split.csv", index=False)
    aug_val_df.to_csv(  output_dir / "val_split.csv",   index=False)
    aug_test_df.to_csv( output_dir / "test_split.csv",  index=False)

    print(f"  Train images : {len(aug_train_df)}")
    print(f"  Val images   : {len(aug_val_df)}")
    print(f"  Test images  : {len(aug_test_df)}")
    print(f"  CSVs written to {output_dir}")
    print(f"\n  Class distribution (train, after aug):")
    print(aug_train_df['diagnosis'].value_counts().sort_index().to_string())
    print(f"  DATASET_DIR = '{output_dir}'")

In [8]:
if __name__ == "__main__":
    main()


Dataset root : /kaggle/input/datasets/almiraraisa/aptos-2019-224pixels
Output root  : /kaggle/working/aptos-2019-augmented
Train rows   : 2563  →  15378 after augmentation
Val rows     : 549
Test rows    : 550



Copying test: 100%|██████████| 550/550 [00:02<00:00, 221.85it/s]

  Train images : 15378
  Val images   : 549
  Test images  : 550
  CSVs written to /kaggle/working/aptos-2019-augmented

  Class distribution (train, after aug):
diagnosis
0    7578
1    1554
2    4194
3     810
4    1242
  DATASET_DIR = '/kaggle/working/aptos-2019-augmented'


In [11]:
output_dir = Path("/kaggle/working/aptos-2019-augmented")
zip_path = output_dir.parent / "aptos_augmented.zip"
print(f"\n  Zipping output to {zip_path} ...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in output_dir.rglob("*"):
        if file.is_file():
            zf.write(file, file.relative_to(output_dir))
print(f"  Done. Download: {zip_path}")


  Zipping output to /kaggle/working/aptos_augmented.zip ...
  Done. Download: /kaggle/working/aptos_augmented.zip


In [13]:
from IPython.display import FileLink
FileLink(r'aptos_augmented.zip')

/kaggle/working/aptos_augmented.zip